### Anchor H5py Generator

This notebook is exclusively for generating anchors (or any galaxy) that cannot be found in the SGA 2025 database (which have h5s attached)

In [21]:
import h5py # Read and write h5 files
from astropy.table import Table
from astropy.io import fits # Read fits files

import numpy as np
import pandas as pd # Work with dataframes
from pathlib import Path
from tqdm.notebook import tqdm # Progress bar

from concurrent.futures import ProcessPoolExecutor # Allow for concurrent work

In [6]:
# ============================================================
# USER CONFIGURATION
# ============================================================ 
output_dir = Path("/pscratch/sd/q/qshimp/Sorter")
anchor_path = Path("/pscratch/sd/q/qshimp/SGA2020-data/Anchors/VI_4000_sga152x152_complete.csv")
name = "Julia"

npix = 152
batch_size = 64
chunk_size = 1000
workers = 16  # ProcessPoolExecutor x16 -- best config from h5py_efficiency.ipynb (528.3 gal/s)
verbose = 4

# IMPORTANT: CHANGE TO MATCH YOUR DATA
anchor_columns = {
    # Change  -  Don't change
    "ra":       "target_ra",
    "dec":      "target_dec",
    "Main_type":"Main_Type",
    "ref_id":   "targetid",
    "Path":     "Path"
}

In [24]:
def read_catalog(path):  # Read in anchors
    path = Path(path)
    if path.suffix == ".csv":
        return pd.read_csv(path)
    if path.suffix == ".fits":
        return Table.read(path, hdu=1).to_pandas()
    raise ValueError(f"Unsupported file type: {path.suffix}")

def read_one(args): # Read a galaxy and return its data
    (i, path_str, npix, ra_i, dec_i, main_type_i, targetid_i) = args

    path = Path(path_str)
    img_data = None
 
    try:
        if not path.exists():
            return None
        with fits.open(path, memmap=True, ignore_missing_simple=True, ignore_missing_end=True) as hdul:
            img_data = hdul[0].data.copy()
 
    except Exception as e:
        print(f"Unexpected error reading index {i} ({path}): {e}")
        return None
 
    if img_data is None or img_data.shape != (3, npix, npix):
        return None
 
    return {
        "images": img_data,
        "ra": float(ra_i),
        "dec": float(dec_i),
        "Main_Type": int(main_type_i),
        "targetid": int(targetid_i),
    }

In [28]:
def create_new_h5(catalog, output_path, verbose):
    # Get data
    n_galaxies = len(catalog)
    ra = catalog["target_ra"].to_numpy()
    dec = catalog["target_dec"].to_numpy()
    main_type = catalog["Main_Type"].to_numpy(np.int32)
    targetid = catalog["targetid"].to_numpy(np.int64)
    paths = [str(p).strip() for p in catalog["Path"]]
 
    # Precomputed per-galaxy arg tuples 
    all_args = [
        (i, paths[i], npix, ra[i], dec[i], main_type[i], targetid[i])
        for i in range(n_galaxies)
    ]

    # Retrieve existing file
    if output_path.exists():
        if verbose:
            print(f"Resuming from {output_path}")
 
        f = h5py.File(output_path, "r+")
        valid_entries = int(f.attrs.get("valid_entries", 0))
        if valid_entries > 0:
            if not np.array_equal(f["targetid"][:valid_entries], targetid[:valid_entries]):
                raise ValueError("Existing HDF5 does not match this catalog.")
     # Create new file
    else:
        f = h5py.File(output_path, "w")
        valid_entries = 0
 
    with f:
        datasets = {}
        if valid_entries == 0:
            datasets["images"] = f.create_dataset(
                "images",
                (n_galaxies, 3, npix, npix),
                dtype="f4",
                chunks=(batch_size, 3, npix, npix),
            )
            for dset_name, dtype in {"ra": "f8", "dec": "f8", "Main_Type": "i4", "targetid": "i8"}.items():
                datasets[dset_name] = f.create_dataset(dset_name, (n_galaxies,), dtype=dtype)
            f.flush()
        else:
            for dset_name in ["images", "ra", "dec", "Main_Type", "targetid"]:
                datasets[dset_name] = f[dset_name]
 
        batch = []
        n_failed = 0
        
        def write_batch(): # Flush the current batch list of result dicts to the datasets.
            nonlocal valid_entries, batch
            n = len(batch)
            if n == 0:
                return
            s = slice(valid_entries, valid_entries + n)
            images = np.stack([b["images"] for b in batch])
            datasets["images"][s] = images
            datasets["ra"][s] = np.fromiter((b["ra"] for b in batch), np.float64)
            datasets["dec"][s] = np.fromiter((b["dec"] for b in batch), np.float64)
            datasets["Main_Type"][s] = np.fromiter((b["Main_Type"] for b in batch), np.int32)
            datasets["targetid"][s] = np.fromiter((b["targetid"] for b in batch), np.int64)
            valid_entries += n
            f.attrs["valid_entries"] = valid_entries
            pbar.update(n)
            batch = []
 
        with ProcessPoolExecutor(max_workers=workers) as executor:
            with tqdm(total=n_galaxies, initial=valid_entries) as pbar:
                
                for start in range(valid_entries, n_galaxies, chunk_size):
                    end = min(start + chunk_size, n_galaxies)
                    for result in executor.map(read_one, all_args[start:end], chunksize=16):
                        if result is None:
                            n_failed += 1
                            pbar.update(1)
                            continue
                        
                        batch.append(result)
                        if len(batch) >= batch_size:
                            write_batch()
                            
                    f.flush()
                
        # Final partial batch, if any.
        write_batch()
        f.flush()
 
    if verbose:
        print(f"Saved {valid_entries} galaxies to {output_path} ({n_failed} failed/skipped)")
        print(catalog.columns)
    return output_path

In [30]:
run_path = output_dir / "binary_classifier" / name
run_path.mkdir(parents=True, exist_ok=True)

anchor_df = read_catalog(anchor_path)
anchor_df = (anchor_df[list(anchor_columns.keys())].rename(columns=anchor_columns))

dir = run_path / "h5_files"
dir.mkdir(parents=True, exist_ok=True)
path = dir / f"{name}.h5"
path = create_new_h5(targets, path, verbose)

  0%|          | 0/4000 [00:00<?, ?it/s]

Saved 4000 galaxies to /pscratch/sd/q/qshimp/Sorter/binary_classifier/Julia/h5_files/Julia.h5 (0 failed/skipped)
Index(['target_ra', 'target_dec', 'Main_Type', 'targetid', 'Path'], dtype='str')
